In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv('/kaggle/input/forest-cover-type-prediction/train.csv').sample(5000)
train

In [ ]:
train2 = train.drop(columns = ['Cover_Type','Id'])
train2

In [ ]:
import xgboost as xgb

# logloss
# dataframe 형태로 출력
params = {"learning_rate": 0.1, "objective": "multi:softmax", "num_class": 7, "eval_metric": "mlogloss"}
xgb_train = xgb.DMatrix(train2, train["Cover_Type"]-1)  # data matrix

cv_result = xgb.cv(params, xgb_train, 10000, stratified = True, nfold = 3, early_stopping_rounds = 50)
cv_result["test-mlogloss-mean"].min()

In [ ]:
# xgb
# BayesianOptimization
def xgb_bayesian(colsample_bytree, max_depth, subsample):
    params = {"learning_rate": 0.1, "objective": "multi:softmax", "num_class":7, "eval_metric": "mlogloss", 
              "colsample_bytree": colsample_bytree, "max_depth": int(max_depth), "subsample": subsample}
    xgb_train = xgb.DMatrix(train2, train["Cover_Type"]-1)
    cv_result = xgb.cv(params, xgb_train, 10000, stratified = True, nfold = 3, early_stopping_rounds = 50)
    return -cv_result["test-mlogloss-mean"].min()
                    
from bayes_opt import BayesianOptimization
bo = BayesianOptimization(xgb_bayesian, {"colsample_bytree": (0.2, 0.8), "max_depth": (4,14), "subsample": (0.5, 0.9)})


bo.maximize(init_points = 10, n_iter = 30)

In [ ]:
#################### 미완료 대회